# CASIA-WebFace dataset stats

Step 1.1: read the identity label of every record in `train.rec`.

Only the small header at the start of each record is read, so no images are decoded. Records with
flag 0 are face images, and their label is the identity number. Records with any other flag are
dataset bookkeeping (key 0 and one record per identity listing its image range).

Run this notebook from the `Research` folder so `lib` can be imported.

In [4]:
from pathlib import Path
import time

from lib.dataset_reader import DEFAULT_CASIA_DIRECTORY, iter_casia_labels

dataset_directory = DEFAULT_CASIA_DIRECTORY
print(dataset_directory)

C:\Users\Fawaz\OneDrive\Desktop\Fernas\Research\data\casia-webface


In [5]:
start_time = time.time()
records = list(iter_casia_labels(dataset_directory))
print(f"Read {len(records)} record headers in {time.time() - start_time:.1f} s.")

Read 501196 record headers in 1.9 s.


In [6]:
image_records = [(record_key, label) for record_key, flag, label in records if flag == 0]
bookkeeping_records = [record_key for record_key, flag, _ in records if flag != 0]

print(f"{len(image_records)} face images with an identity label")
print(f"{len(bookkeeping_records)} bookkeeping records")
print("First 5 image records (record key, identity label):", image_records[:5])

490623 face images with an identity label
10573 bookkeeping records
First 5 image records (record key, identity label): [(1, 0.0), (2, 0.0), (3, 0.0), (4, 0.0), (5, 0.0)]


## Step 1.2: leave out the bookkeeping records

Only flag 0 records are face images, so `image_records` above already leaves the rest out. The
checks below confirm the bookkeeping records are exactly what this dataset format should have:
key 0 as the dataset header, then one record per identity right after the last image. The
identity count comes from the dataset's `property` file.

In [7]:
property_path = dataset_directory / "property"
identity_count, image_height, image_width = (int(value) for value in property_path.read_text().split(","))
print(f"property file: {identity_count} identities, {image_width}x{image_height} images")

property file: 10572 identities, 112x112 images


In [8]:
image_keys = sorted(record_key for record_key, _ in image_records)
image_labels = sorted({label for _, label in image_records})
identity_record_keys = sorted(record_key for record_key in bookkeeping_records if record_key != 0)

checks = {
    "key 0 is a bookkeeping record": 0 in bookkeeping_records,
    "one bookkeeping record per identity, plus key 0": len(bookkeeping_records) == identity_count + 1,
    "identity records come right after the last image, with no gaps": identity_record_keys == list(range(len(image_records) + 1, len(image_records) + 1 + identity_count)),
    "image keys run from 1 with no gaps": image_keys == list(range(1, len(image_records) + 1)),
    "identity labels are the whole numbers 0 to identity_count - 1": image_labels == [float(label) for label in range(identity_count)],
}
for description, passed in checks.items():
    print(f"{'OK  ' if passed else 'FAIL'} {description}")

OK   key 0 is a bookkeeping record
OK   one bookkeeping record per identity, plus key 0
OK   identity records come right after the last image, with no gaps
OK   image keys run from 1 with no gaps
OK   identity labels are the whole numbers 0 to identity_count - 1


In [9]:
print(f"Kept {len(image_records)} face images from {len(image_labels)} identities.")
print(f"Left out {len(bookkeeping_records)} bookkeeping records.")

Kept 490623 face images from 10572 identities.
Left out 10573 bookkeeping records.
